In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import sys
import time
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_X_y, check_array
from sklearn.datasets import (
    make_classification,
    load_breast_cancer,
    load_digits,
    load_wine,
    load_iris
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    log_loss,
    jaccard_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings("ignore")

# =========================================================
# RESEARCH PIPELINE
# =========================================================
# Hybrid Design:
# - Code 4 Structure
# - Code 1 Methodology
# =========================================================

# =========================================================
# DATASET LOADING
# =========================================================
def load_datasets():
    datasets = {}
    X1, y1 = make_classification(
        n_samples=10000,
        n_features=20,
        n_informative=15,
        random_state=42
    )
    datasets["Synthetic_10000"] = (X1, y1)
    X2, y2 = make_classification(
        n_samples=5000,
        n_features=20,
        n_informative=15,
        random_state=42
    )
    datasets["Synthetic_5000"] = (X2, y2)
    d = load_breast_cancer()
    datasets["Breast_Cancer"] = (d.data, d.target)
    d = load_digits()
    datasets["Digits"] = (d.data, d.target)
    d = load_wine()
    datasets["Wine"] = (d.data, d.target)
    d = load_iris()
    datasets["Iris"] = (d.data, d.target)
    return datasets

# =========================================================
# HYBRID CONDENSER
# =========================================================
class SVMBoundaryCondenser(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        target_ratio=0.6,
        kernel="linear",
        C=1.0,
        pruning_threshold=0.3,
        use_margin_sampling=True,
        random_state=42
    ):
        self.target_ratio = target_ratio
        self.kernel = kernel
        self.C = C
        self.pruning_threshold = pruning_threshold
        self.use_margin_sampling = use_margin_sampling
        self.random_state = random_state

    def fit(self, X, y):
        np.random.seed(self.random_state)
        X, y = check_X_y(X, y)
        self.original_size_ = len(X)
        self.scaler_ = StandardScaler()
        X_scaled = self.scaler_.fit_transform(X)
        self.svm_ = SVC(
            kernel=self.kernel,
            C=self.C,
            probability=True,
            random_state=self.random_state
        )
        self.svm_.fit(X_scaled, y)
        sv_idx = self.svm_.support_
        sv_probs = self.svm_.predict_proba(X_scaled[sv_idx])
        conf = np.max(sv_probs, axis=1)
        mask = conf > self.pruning_threshold
        if np.sum(mask) > 0:
            clean_sv_idx = sv_idx[mask]
        else:
            clean_sv_idx = sv_idx
        X_sv = X[clean_sv_idx]
        y_sv = y[clean_sv_idx]
        all_idx = np.arange(len(X))
        non_sv_idx = np.setdiff1d(all_idx, sv_idx)
        X_non_sv = X[non_sv_idx]
        y_non_sv = y[non_sv_idx]
        target_size = int(len(X) * self.target_ratio)
        remaining_budget = target_size - len(X_sv)
        if remaining_budget > 0 and len(non_sv_idx) > 0:
            decision_values = self.svm_.decision_function(X_scaled[non_sv_idx])
            if decision_values.ndim == 2:
                distances = np.min(np.abs(decision_values), axis=1)
            else:
                distances = np.abs(decision_values)
            distances = distances + 1e-8
            probs = 1.0 / distances
            probs[np.isnan(probs)] = 0
            probs[np.isinf(probs)] = 0
            total = probs.sum()
            if total == 0:
                probs = np.ones_like(probs) / len(probs)
            else:
                probs = probs / total
            sample_size = min(remaining_budget, len(non_sv_idx))
            if self.use_margin_sampling:
                chosen = np.random.choice(len(non_sv_idx), size=sample_size, replace=False, p=probs)
            else:
                chosen = np.random.choice(len(non_sv_idx), size=sample_size, replace=False)
            X_sampled = X_non_sv[chosen]
            y_sampled = y_non_sv[chosen]
        else:
            X_sampled = np.empty((0, X.shape[1]))
            y_sampled = np.array([])
        self.X_condensed_ = np.vstack([X_sv, X_sampled])
        self.y_condensed_ = np.concatenate([y_sv, y_sampled])
        self.condensation_stats_ = {
            "original_size": len(X),
            "condensed_size": len(self.X_condensed_),
            "compression_ratio": len(X) / len(self.X_condensed_),
            "reduction_percent": (1 - len(self.X_condensed_) / len(X)) * 100,
            "n_support_vectors": len(X_sv),
            "n_sampled_points": len(X_sampled)
        }
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_array(X)
        return self.X_condensed_

    def get_condensed_data(self):
        return (self.X_condensed_, self.y_condensed_)

# =========================================================
# COMPLEXITY MEASUREMENT (FLOPS PROXY)
# =========================================================
def measure_complexity(model, X_test):
    """
    Measures model size in bytes and average inference time per sample.
    These act as proxies for 'heaviness' (FLOPS) in traditional ML.
    """
    # Model Size in memory
    model_size = sys.getsizeof(model)
    
    # Average Inference Time
    start_time = time.time()
    model.predict(X_test)
    end_time = time.time()
    inf_time_per_sample = (end_time - start_time) / len(X_test)
    
    return model_size, inf_time_per_sample

# =========================================================
# EVALUATION FUNCTION
# =========================================================
def evaluate_pipeline(condenser, model, X, y, cv=5):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    
    # Metrics containers
    metrics_full = {
        "acc": [], "prec": [], "rec": [], "f1": [], "jacc": [], "loss": [], "size": [], "time": []
    }
    metrics_cond = {
        "acc": [], "prec": [], "rec": [], "f1": [], "jacc": [], "loss": [], "size": [], "time": []
    }
    compression_ratios = []
    
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # FULL DATASET
        scaler_full = StandardScaler()
        X_train_f = scaler_full.fit_transform(X_train)
        X_test_f = scaler_full.transform(X_test)
        
        full_model = model.__class__(**model.get_params())
        full_model.fit(X_train_f, y_train)
        
        pred_f = full_model.predict(X_test_f)
        prob_f = full_model.predict_proba(X_test_f)
        
        metrics_full["acc"].append(accuracy_score(y_test, pred_f))
        metrics_full["prec"].append(precision_score(y_test, pred_f, average='weighted'))
        metrics_full["rec"].append(recall_score(y_test, pred_f, average='weighted'))
        metrics_full["f1"].append(f1_score(y_test, pred_f, average='weighted'))
        metrics_full["jacc"].append(jaccard_score(y_test, pred_f, average='weighted'))
        metrics_full["loss"].append(log_loss(y_test, prob_f))
        
        f_size, f_time = measure_complexity(full_model, X_test_f)
        metrics_full["size"].append(f_size)
        metrics_full["time"].append(f_time)

        # CONDENSED DATASET
        cond = condenser.__class__(**condenser.get_params())
        cond.fit(X_train, y_train)
        X_c, y_c = cond.get_condensed_data()
        compression_ratios.append(cond.condensation_stats_["compression_ratio"])
        
        scaler_cond = StandardScaler()
        X_c_scaled = scaler_cond.fit_transform(X_c)
        X_test_c_scaled = scaler_cond.transform(X_test)
        
        cond_model = model.__class__(**model.get_params())
        cond_model.fit(X_c_scaled, y_c)
        
        pred_c = cond_model.predict(X_test_c_scaled)
        prob_c = cond_model.predict_proba(X_test_c_scaled)
        
        metrics_cond["acc"].append(accuracy_score(y_test, pred_c))
        metrics_cond["prec"].append(precision_score(y_test, pred_c, average='weighted'))
        metrics_cond["rec"].append(recall_score(y_test, pred_c, average='weighted'))
        metrics_cond["f1"].append(f1_score(y_test, pred_c, average='weighted'))
        metrics_cond["jacc"].append(jaccard_score(y_test, pred_c, average='weighted'))
        metrics_cond["loss"].append(log_loss(y_test, prob_c))
        
        c_size, c_time = measure_complexity(cond_model, X_test_c_scaled)
        metrics_cond["size"].append(c_size)
        metrics_cond["time"].append(c_time)

    # Final Averages
    res = {}
    for m in metrics_full.keys():
        res[f"mean_full_{m}"] = np.mean(metrics_full[m])
        res[f"mean_cond_{m}"] = np.mean(metrics_cond[m])
    
    res["mean_difference"] = np.mean(metrics_cond["acc"]) - np.mean(metrics_full["acc"])
    t_stat, p_val = stats.ttest_rel(metrics_cond["acc"], metrics_full["acc"])
    
    res.update({
        "p_value": p_val,
        "t_statistic": t_stat,
        "compression_ratio": np.mean(compression_ratios),
        "significant": p_val < 0.05
    })
    return res

# =========================================================
# PRINT REPORT
# =========================================================
def print_report(dataset_name, model_name, results):
    print("\n" + "=" * 70)
    print(f"DATASET: {dataset_name} | MODEL: {model_name}")
    print("=" * 70)
    print(f"Accuracy:    Full {results['mean_full_acc']:.4f} | Cond {results['mean_cond_acc']:.4f} | Diff: {results['mean_difference']:+.4f}")
    print(f"Precision:   Full {results['mean_full_prec']:.4f} | Cond {results['mean_cond_prec']:.4f}")
    print(f"Recall:      Full {results['mean_full_rec']:.4f} | Cond {results['mean_cond_rec']:.4f}")
    print(f"F1 Score:    Full {results['mean_full_f1']:.4f} | Cond {results['mean_cond_f1']:.4f}")
    print(f"Jaccard:     Full {results['mean_full_jacc']:.4f} | Cond {results['mean_cond_jacc']:.4f}")
    print(f"Log Loss:    Full {results['mean_full_loss']:.4f} | Cond {results['mean_cond_loss']:.4f}")
    print("\n--- Model Heaviness (Complexity Proxy) ---")
    print(f"Model Size:  Full {results['mean_full_size']} bytes | Cond {results['mean_cond_size']} bytes")
    print(f"Inf Time:    Full {results['mean_full_time']:.6f}s | Cond {results['mean_cond_time']:.6f}s")
    print("\n--- Statistical Validation ---")
    print(f"p-value: {results['p_value']:.6f} | Ratio: {results['compression_ratio']:.2f}x")
    print(f"Test: {'SIGNIFICANT' if results['significant'] else 'NOT SIGNIFICANT'}")

# =========================================================
# MAIN EXPERIMENT
# =========================================================
def run_experiment():
    datasets = load_datasets()
    models = {
        "LogisticRegression": LogisticRegression(max_iter=1000),
        "KNN": KNeighborsClassifier(),
        "NaiveBayes": GaussianNB(),
        "SVM": SVC(probability=True, random_state=42),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
        "GradientBoosting": GradientBoostingClassifier(random_state=42)
    }
    
    condenser = SVMBoundaryCondenser(target_ratio=0.6, random_state=42)
    final_results = []
    
    for dataset_name, (X, y) in datasets.items():
        print("\n" + "#" * 70)
        print(f"RUNNING DATASET: {dataset_name}")
        print("#" * 70)
        for model_name, model in models.items():
            results = evaluate_pipeline(condenser, model, X, y)
            print_report(dataset_name, model_name, results)
            
            # Store for summary
            final_results.append({
                "Dataset": dataset_name, "Model": model_name,
                "Full_Acc": results['mean_full_acc'], "Cond_Acc": results['mean_cond_acc'],
                "Diff": results['mean_difference'], "Compression": results['compression_ratio'],
                "Model_Size_Diff": results['mean_cond_size'] - results['mean_full_size']
            })

    df = pd.DataFrame(final_results)
    print("\n" + "=" * 100)
    print("FINAL RESEARCH SUMMARY")
    print("=" * 100)
    print(df.to_string(index=False))
    df.to_csv("research_results_expanded.csv", index=False)

if __name__ == "__main__":
    print("=" * 70)
    print("BOUNDARY-AWARE DATASET CONDENSATION: EXPANDED METRICS")
    print("=" * 70)
    run_experiment()

BOUNDARY-AWARE DATASET CONDENSATION: EXPANDED METRICS

######################################################################
RUNNING DATASET: Synthetic_10000
######################################################################

DATASET: Synthetic_10000 | MODEL: LogisticRegression
Accuracy:    Full 0.8196 | Cond 0.8189 | Diff: -0.0007
Precision:   Full 0.8203 | Cond 0.8195
Recall:      Full 0.8196 | Cond 0.8189
F1 Score:    Full 0.8195 | Cond 0.8188
Jaccard:     Full 0.6943 | Cond 0.6933
Log Loss:    Full 0.4062 | Cond 0.4311

--- Model Heaviness (Complexity Proxy) ---
Model Size:  Full 48.0 bytes | Cond 48.0 bytes
Inf Time:    Full 0.000000s | Cond 0.000000s

--- Statistical Validation ---
p-value: 0.051606 | Ratio: 1.67x
Test: NOT SIGNIFICANT

DATASET: Synthetic_10000 | MODEL: KNN
Accuracy:    Full 0.9595 | Cond 0.9489 | Diff: -0.0106
Precision:   Full 0.9597 | Cond 0.9491
Recall:      Full 0.9595 | Cond 0.9489
F1 Score:    Full 0.9595 | Cond 0.9489
Jaccard:     Full 0.9222 | Cond 